# OOP Part 3

## The Copy Assignment Operator; The Rule of Three; Function Overloading; Initialization Lists; Operator Overloading

<br><br><br><br><br>
<br><br><br><br><br>

# 1. The Copy Assignment Operator

### We saw that classes with pointer members that point to heap-allocated memory need a custom **destructor** and a custom **copy constructor** to avoid shallow copy bugs and memory leaks. There is one more special member function we need: the **copy assignment operator**.

### First, let's recall the difference between *initializing* an object and *assigning* to an existing object.

In [ ]:
////copyVsAssign.cpp
// Example 1a - Copy constructor vs. copy assignment
#include<iostream>
#include<string>

class Character{
public:
    std::string name_;
    int lives_, coins_;
    int* history_;
    int history_count_;

    Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
        this->history_ = new int[10];
        this->history_count_ = 0;
        for(int i = 0; i < 10; i++) this->history_[i] = 0;
    }
    
    // Deep copy constructor
    Character(const Character& other){
        std::cout << "Copy constructor called\n";
        this->name_ = other.name_;
        this->lives_ = other.lives_;
        this->coins_ = other.coins_;
        this->history_count_ = other.history_count_;
        this->history_ = new int[10];  // allocate NEW memory
        for(int i = 0; i < 10; i++) this->history_[i] = other.history_[i];
    }

    void collect_coins(int num_coins){
        this->coins_ += num_coins;
        this->history_[this->history_count_] = num_coins;
        this->history_count_++;
    }
    void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
    }
    ~Character(){
        std::cout << "Destructor for " << this->name_ << "\n";
        delete[] this->history_;
    }
};

int main(){
    Character mario("Mario");
    mario.collect_coins(10);
    
    Character luigi = mario;  // which one is this: copy constructor or assignment?
    
    Character wario("Wario");
    wario = mario;            // which one is this: copy constructor or assignment?
}

<br><br><br><br><br>
<br><br><br><br><br>

### The key difference:
- ### `Character luigi = mario;` creates a **new** object. This calls the **copy constructor**.
- ### `wario = mario;` overwrites an **existing** object. This calls the **copy assignment operator** (`operator=`).

### We wrote a deep copy constructor, but we haven't written a copy assignment operator yet. The default one does a shallow copy — the same problem we saw before!

<br><br><br><br><br>
<br><br><br><br><br>

### Let's see the problem:

In [ ]:
////assignmentBug.cpp
// Example 1b - Default assignment causes the same shallow copy bug
#include<iostream>
#include<string>

class Character{
public:
    std::string name_;
    int lives_, coins_;
    int* history_;
    int history_count_;

    Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
        this->history_ = new int[10];
        this->history_count_ = 0;
        for(int i = 0; i < 10; i++) this->history_[i] = 0;
    }
    Character(const Character& other){
        this->name_ = other.name_;
        this->lives_ = other.lives_;
        this->coins_ = other.coins_;
        this->history_count_ = other.history_count_;
        this->history_ = new int[10];
        for(int i = 0; i < 10; i++) this->history_[i] = other.history_[i];
    }
    void collect_coins(int num_coins){
        this->coins_ += num_coins;
        this->history_[this->history_count_] = num_coins;
        this->history_count_++;
    }
    void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins";
        std::cout << ", history at " << this->history_ << "\n";
    }
    ~Character(){
        std::cout << "Destructor for " << this->name_ << ", history_ at " << this->history_ << "\n";
        delete[] this->history_;
    }
};

int main(){
    Character mario("Mario");
    mario.collect_coins(10);
    
    Character wario("Wario");
    wario.collect_coins(50);
    
    wario = mario;  // default assignment: shallow copy!
    
    mario.display();
    wario.display();  // what address does wario's history_ point to now?
}

<br><br><br><br><br>
<br><br><br><br><br>

### Two problems with the default assignment:
1. ### Wario's old `history_` array is **leaked** — no one will ever `delete[]` it.
2. ### Wario and Mario now **share** the same `history_` array — same double-free bug as before.

<br><br><br><br><br>
<br><br><br><br><br>

### Writing the copy assignment operator:

### The signature is: `Character& operator=(const Character& other)`

### It needs to:
1. ### Clean up the existing resource (delete the old array)
2. ### Copy the data from `other` (deep copy, just like the copy constructor)
3. ### Return `*this` (so we can chain assignments like `a = b = c`)

<br><br><br><br><br>
<br><br><br><br><br>

### One subtle issue: what if the object is accidentally assigned to itself? You might think "who would write `mario = mario`?" — but it can happen through references without you realizing it:

In [ ]:
////selfAssignment.cpp
// Example 1c - How self-assignment can happen accidentally
#include<iostream>
#include<string>

int main(){
    Character mario("Mario");
    mario.collect_coins(10);
    
    Character& ref = mario;  // ref is just another name for mario
    ref = mario;             // self-assignment! ref IS mario
    
    // Without a self-assignment check:
    // Step 1 deletes mario's history_
    // Step 2 tries to copy from mario's history_ — which we just deleted!
}

<br><br><br><br><br>
<br><br><br><br><br>

### So we need to check `if(this != &other)` before doing anything. With that in mind, here's the full implementation:

In [ ]:
////assignmentOperator.cpp
// Example 1c - Writing the copy assignment operator
#include<iostream>
#include<string>

class Character{
public:
    std::string name_;
    int lives_, coins_;
    int* history_;
    int history_count_;

    Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
        this->history_ = new int[10];
        this->history_count_ = 0;
        for(int i = 0; i < 10; i++) this->history_[i] = 0;
    }
    
    // Deep copy constructor
    Character(const Character& other){
        this->name_ = other.name_;
        this->lives_ = other.lives_;
        this->coins_ = other.coins_;
        this->history_count_ = other.history_count_;
        this->history_ = new int[10];
        for(int i = 0; i < 10; i++) this->history_[i] = other.history_[i];
    }
    
    // Copy assignment operator
    Character& operator=(const Character& other){
        if(this != &other){           // 0. check for self-assignment (a = a)
            delete[] this->history_;         // 1. clean up existing resource
            this->name_ = other.name_;       // 2. copy all data
            this->lives_ = other.lives_;
            this->coins_ = other.coins_;
            this->history_count_ = other.history_count_;
            this->history_ = new int[10];    //    allocate NEW memory
            for(int i = 0; i < 10; i++) this->history_[i] = other.history_[i];
        }
        return *this;                  // 3. return *this
    }

    void collect_coins(int num_coins){
        this->coins_ += num_coins;
        this->history_[this->history_count_] = num_coins;
        this->history_count_++;
    }
    void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins";
        std::cout << ", history at " << this->history_ << "\n";
    }
    ~Character(){
        std::cout << "Destructor for " << this->name_ << ", history_ at " << this->history_ << "\n";
        delete[] this->history_;
    }
};

int main(){
    Character mario("Mario");
    mario.collect_coins(10);
    
    Character wario("Wario");
    wario.collect_coins(50);
    
    wario = mario;  // now uses our custom assignment operator!
    
    mario.display();
    wario.display();  // different address now?
}

<br><br><br><br><br>
<br><br><br><br><br>

# 2. The Rule of Three

### The **Rule of Three**: If a class needs a custom version of any one of these, it almost certainly needs all three:
1. ### **Destructor** — cleans up resources
2. ### **Copy constructor** — creates a new object as a deep copy
3. ### **Copy assignment operator** — overwrites an existing object with a deep copy

### When does a class need these? When it has a pointer member that points to memory allocated on the heap (with `new`). A pointer to a stack variable doesn't cause these problems — it's specifically about *owning* heap memory.

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////ruleOfThreeYouTry.cpp
// Example 2a - You try! Does this class need a custom destructor, copy constructor,
// and/or copy assignment operator? Why or why not?
#include<iostream>
#include<string>

class Student{
private:
    std::string name_;
    double gpa_;
public:
    Student(std::string n, double g){
        name_ = n;
        gpa_ = g;
    }
    void display() const {
        std::cout << name_ << ": " << gpa_ << "\n";
    }
};

// What about this class?
class Roster{
private:
    std::string* names_;
    int size_;
public:
    Roster(int n){
        size_ = n;
        names_ = new std::string[n];
    }
    void setName(int i, std::string name){
        names_[i] = name;
    }
    void display() const {
        for(int i = 0; i < size_; i++)
            std::cout << names_[i] << "\n";
    }
};

int main(){
    Student s1("Alice", 3.8);
    Student s2 = s1;  // is this safe?
    
    Roster r1(3);
    r1.setName(0, "Alice");
    Roster r2 = r1;   // is this safe?
}

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////ruleOfThreeRoster.cpp
// Example 2b - You try! Add the Rule of Three to the Roster class.
#include<iostream>
#include<string>

class Roster{
private:
    std::string* names_;
    int size_;
public:
    Roster(int n){
        size_ = n;
        names_ = new std::string[n];
    }
    
    // write the destructor
    
    
    // write the copy constructor
    
    
    // write the copy assignment operator
    
    
    void setName(int i, std::string name){
        names_[i] = name;
    }
    void display() const {
        for(int i = 0; i < size_; i++)
            std::cout << names_[i] << "\n";
    }
};

int main(){
    Roster r1(3);
    r1.setName(0, "Alice");
    r1.setName(1, "Bob");
    r1.setName(2, "Carol");
    
    Roster r2 = r1;       // should use copy constructor
    r2.setName(0, "Zara"); // should NOT affect r1
    
    Roster r3(1);
    r3 = r1;              // should use copy assignment
    
    std::cout << "r1: "; r1.display();
    std::cout << "r2: "; r2.display();
    std::cout << "r3: "; r3.display();
}

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////heapReview.cpp
// Example 2c - You try! Draw the memory picture. 
// Where is the Character object? Where is its history_ array?
// What happens when we call delete ptr?
#include<iostream>
#include<string>

class Character{
public:
    std::string name_;
    int coins_;
    int* history_;
    int history_count_;

    Character(std::string n){
        name_ = n;
        coins_ = 0;
        history_count_ = 0;
        history_ = new int[10];
        for(int i = 0; i < 10; i++) history_[i] = 0;
    }
    void collect_coins(int num_coins){
        coins_ += num_coins;
        history_[history_count_] = num_coins;
        history_count_++;
    }
    ~Character(){
        std::cout << "Destructor called for " << name_ << "\n";
        delete[] history_;
    }
};

int main(){
    Character* ptr = new Character("Mario");  // where does the object live?
    ptr->collect_coins(10);
    
    // How many things are on the heap right now?
    // If we call delete ptr, what happens to history_?
    
    delete ptr;
}

<br><br><br><br><br>
<br><br><br><br><br>

# 3. Function Overloading and Constructor Overloading

### In C++, you can have multiple functions with the **same name** as long as they have **different parameter lists**. The compiler figures out which one to call based on the arguments you pass. This is called **function overloading**.

In [ ]:
////functionOverloading.cpp
// Example 3a - Function overloading
#include<iostream>
#include<string>

void print(int x){
    std::cout << "int: " << x << "\n";
}
void print(double x){
    std::cout << "double: " << x << "\n";
}
void print(std::string x){
    std::cout << "string: " << x << "\n";
}

int main(){
    print(5);
    print(3.14);
    print("hello");
}

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////constructorOverloading.cpp
// Example 3b - Constructor overloading
#include<iostream>

class Fraction{
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}   // two parameters
    Fraction(int n) : num_(n), den_(1) {}           // one parameter (whole number)
    
    void display() const {
        std::cout << num_ << "/" << den_ << "\n";
    }
};

int main(){
    Fraction a(3, 4);   // calls the two-parameter constructor
    Fraction b(5);      // calls the one-parameter constructor
    a.display();
    b.display();
}

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////initializerLists.cpp
// Example 4a - Two ways to write a constructor
#include<iostream>
#include<string>

class Character{
public:
    std::string name_;
    int lives_;
    int coins_;
    
    // Style 1: Assignment in the constructor body
    // Character(std::string n){
    //     name_ = n;
    //     lives_ = 3;
    //     coins_ = 0;
    // }
    
    // Style 2: Initialization list (preferred in C++)
    Character(std::string n) : name_(n), lives_(3), coins_(0) {}
    
    void display() const {
        std::cout << name_ << ": " << lives_ << " lives, " << coins_ << " coins\n";
    }
};

int main(){
    Character mario("Mario");
    mario.display();
}

<br><br><br><br><br>
<br><br><br><br><br>

### Both styles do the same thing for simple types. But initialization lists are preferred in C++ because:
- ### They are slightly more efficient (initialize directly, rather than default-construct then assign)
- ### They are **required** for `const` members and reference members (which can't be assigned to after creation)

### You'll see this syntax used throughout the rest of the course. The format is:
### `ClassName(params) : member1(value1), member2(value2) { /* body */ }`

<br><br><br><br><br>
<br><br><br><br><br>

# 5. Operator Overloading

### We wrote `operator=` for the Character class in section 1. The `=` sign is an **operator**, and by writing our own version of it, we **overloaded** it. In C++, we can overload many operators to work with our own classes. In fact, the idea of **overloading** is the same concept we just saw with functions — same name, different behavior depending on the types.

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////fractionClass.cpp
// Example 5a - A Fraction class
#include<iostream>

class Fraction{
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}
    
    int getNum() const { return num_; }
    int getDen() const { return den_; }
    
    void display() const {
        std::cout << num_ << "/" << den_;
    }
};

int main(){
    Fraction a(1, 2);
    Fraction b(3, 4);
    a.display();
    std::cout << " + ";
    b.display();
    std::cout << "\n";
    // Wouldn't it be nice to write: std::cout << a + b << "\n"; ?
}

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////fractionAdd.cpp
// Example 5b - Overloading +
#include<iostream>

class Fraction{
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}
    
    int getNum() const { return num_; }
    int getDen() const { return den_; }
    
    Fraction operator+(const Fraction& other) const {
        int newNum = num_ * other.den_ + other.num_ * den_;
        int newDen = den_ * other.den_;
        return Fraction(newNum, newDen);
    }
    
    void display() const {
        std::cout << num_ << "/" << den_;
    }
};

int main(){
    Fraction a(1, 2);
    Fraction b(3, 4);
    Fraction c = a + b;  // calls a.operator+(b)
    c.display();  // what should this print?
    std::cout << "\n";
}

<br><br><br><br><br>
<br><br><br><br><br>

### Notice: `a + b` is really just a nicer way of writing `a.operator+(b)`. The left operand is `this`, the right operand is `other`.

### Also notice: `operator+` returns a **new** Fraction by value. It does not modify `a` or `b`. That's why the method is `const`.

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////fractionMultiply.cpp
// Example 5c - You try! Add operator* to this class.
#include<iostream>

class Fraction{
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}
    
    Fraction operator+(const Fraction& other) const {
        int newNum = num_ * other.den_ + other.num_ * den_;
        int newDen = den_ * other.den_;
        return Fraction(newNum, newDen);
    }
    
    // write operator* here
    
    
    void display() const {
        std::cout << num_ << "/" << den_;
    }
};

int main(){
    Fraction a(1, 2);
    Fraction b(3, 4);
    Fraction c = a * b;
    c.display();  // should print 3/8
    std::cout << "\n";
}

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////fractionEqual.cpp
// Example 5d - Overloading ==
#include<iostream>

class Fraction{
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}
    
    bool operator==(const Fraction& other) const {
        return num_ * other.den_ == other.num_ * den_;
    }
    
    void display() const {
        std::cout << num_ << "/" << den_;
    }
};

int main(){
    Fraction a(1, 2);
    Fraction b(2, 4);
    Fraction c(3, 4);
    
    std::cout << (a == b) << "\n";  // what prints?
    std::cout << (a == c) << "\n";  // what prints?
}

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////fractionStream.cpp
// Example 5e - Overloading << with a friend function
#include<iostream>

class Fraction{
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}
    
    Fraction operator+(const Fraction& other) const {
        int newNum = num_ * other.den_ + other.num_ * den_;
        int newDen = den_ * other.den_;
        return Fraction(newNum, newDen);
    }
    
    bool operator==(const Fraction& other) const {
        return num_ * other.den_ == other.num_ * den_;
    }
    
    // Declare the friend function inside the class
    friend std::ostream& operator<<(std::ostream& os, const Fraction& f);
};

// Define the friend function outside the class
std::ostream& operator<<(std::ostream& os, const Fraction& f){
    os << f.num_ << "/" << f.den_;
    return os;
}

int main(){
    Fraction a(1, 2);
    Fraction b(3, 4);
    std::cout << a << " + " << b << " = " << a + b << "\n";
}

<br><br><br><br><br>
<br><br><br><br><br>

### Why does `operator<<` return `std::ostream&`? So we can chain calls:
### `std::cout << a << " + " << b` works because each `<<` returns the same `ostream`, so the next `<<` can use it.

### Why is it a `friend`? Because it's not a member function of Fraction, but it needs to access `num_` and `den_` (which are private).

<br><br><br><br><br>
<br><br><br><br><br>

### You try! Predict the output of the following code.

In [ ]:
////fractionPredict.cpp
// Example 5f - You try! Predict the output.
// Assume the Fraction class from Example 5e (with +, ==, and << defined)

int main(){
    Fraction x(1, 3);
    Fraction y(1, 6);
    Fraction z = x + y;
    
    std::cout << z << "\n";
    std::cout << (z == Fraction(1, 2)) << "\n";
    std::cout << (z == Fraction(9, 18)) << "\n";
}

<br><br><br><br><br>
<br><br><br><br><br>

### Summary

### **Rule of Three:** If your class needs a custom destructor, copy constructor, or copy assignment operator, it probably needs all three. This applies when your class manages a resource (like heap memory with `new`).

### **Operator Overloading:** You can define how operators like `+`, `*`, `==`, and `<<` work for your own classes.
- ### Arithmetic and comparison operators are written as member functions: `Fraction operator+(const Fraction& other) const`
- ### The `<<` operator is written as a **friend function** because the left operand is `ostream`, not your class